In [58]:
from concurrent.futures import ThreadPoolExecutor
from dotenv import load_dotenv
from evaluation_utils import map_progress, RAGWithUsage
from ingest import load_faq_data, build_index
from openai import OpenAI
from rag_helper import RAGBase
import pandas as pd

# RAG evals

In [38]:
df_ground_truth = pd.read_csv("/workspaces/llm-zoomcamp-2026-code/Lesson 4B RAG and agent evaluation/data/ground-truth-data-llm-zoomcamp.csv")

In [ ]:
ground_truth = df_ground_truth.to_dict(orient = "records")

In [41]:
documents = load_faq_data()
documents_llm = []
for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [61]:
doc_idx = {}
for doc in documents_llm:
    doc_idx[doc["id"]] = doc
    
# Orphaned rows wegfilteren
ground_truth = [q for q in ground_truth if q["document"] in doc_idx]

In [62]:
q = ground_truth[10]

In [63]:
q

{'question': 'I just found this course. Am I still allowed to join now, and does that affect my chance to get a certificate?',
 'course': 'llm-zoomcamp',
 'document': '74eb249bbf'}

In [64]:
doc_idx[q["document"]]

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

In [65]:
load_dotenv()
openai_client = OpenAI()

In [66]:
assistant = RAGWithUsage(
    index = index,
    llm_client = openai_client,
    course = "llm-zoomcamp"
)

In [67]:
answer = assistant.rag(q["question"])

In [68]:
assistant.total_cost()

0.00057675

In [69]:
doc_id = q["document"]
original_doc = doc_idx[doc_id]
answer_orig = original_doc["answer"]

answer_orig

'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'

In [70]:
rag_result = {
    "question": q["question"],
    "answer": answer,
    "answer_orig": answer_orig,
    "document": doc_id
}
rag_result

{'question': 'I just found this course. Am I still allowed to join now, and does that affect my chance to get a certificate?',
 'answer': 'Yes, you can still join now. But to get a certificate, you must submit your project while the course is still accepting submissions and finish the course with the live cohort.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

In [71]:
def generate_rag_answer(rec):
    question = rec["question"]
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]
    
    answer_llm = assistant.rag(question)
    answer_orig = original_doc["answer"]

    return {
        "question": question,
        "answer": answer_llm,
        "answer_orig": answer_orig,
        "document": doc_id
    }

In [72]:
record = generate_rag_answer(q)
record

{'question': 'I just found this course. Am I still allowed to join now, and does that affect my chance to get a certificate?',
 'answer': 'Yes, you can still join now. But if you want a certificate, you need to finish the course with the live cohort and submit your capstone project while submissions are still being accepted. If you join too late or follow it in self-paced mode, you won’t get a certificate.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

In [73]:
assistant.total_cost()

0.001257

In [74]:
assistant.reset_usage()

In [ ]:
# Costs 0.30 euros
# with ThreadPoolExecutor(max_workers=6) as pool:
#     results = map_progress(pool, ground_truth, generate_rag_answer)

  0%|          | 0/315 [00:00<?, ?it/s]

In [76]:
results

[{'question': 'Can I take this course at my own pace and still receive a certificate at the end?',
  'answer': 'No. You can only get a certificate if you finish the course with a live cohort; certificates are not awarded for self-paced mode.',
  'answer_orig': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.',
  'document': '69d122f12e'},
 {'question': 'Is a certificate available if I complete the course without joining a live cohort?',
  'answer': 'No. You can only get a certificate if you finish the course with a live cohort. Self-paced completion does not qualify for a certificate.',
  'answer_orig': 'No, you can only get a certificate if you finish the course with a "live" cohor

In [78]:
assistant.total_cost()

0.30399825

In [77]:
df_results = pd.DataFrame(results)

In [79]:
df_results.to_csv("data/rag-answers-new.csv", index=False)